In [18]:
import os
import ast

if 'PATH_SET' not in locals():
    os.chdir('../..')
    PATH_SET = True
    
import torch    

if 'THREADS_SET' not in locals():
    try:
        os.environ['OMP_NUM_THREADS'] = '1'
        os.environ['MKL_NUM_THREADS'] = '1'
        
        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)
        THREADS_SET = True
    except:
        pass

import numpy as np
import xarray as xr
import pandas as pd
from scipy.interpolate import RegularGridInterpolator

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('https://raw.githubusercontent.com/dominik-strutz/dotfiles/main/mystyle_background.mplstyle')

In [19]:
from helpers.geographic_setup import (
    design_space_shoulder,
    events_shoulder,
    picking_stats_shoulder,
    topo_data,
)

from helpers.helper_functions import concave_hull2D_prior_dist_constructor
from helpers.likelihood import DataLikelihood
from helpers.forward import TTLookup

In [20]:
vp_model_shoulder = xr.load_dataarray(
    'data/velocity_model/3d_model_final/vp_xarray.nc',
    engine='netcdf4',
    format='NETCDF4'
    )

In [21]:
SHOULDER_E_MIN = vp_model_shoulder['easting'].values.min()
SHOULDER_E_MAX = vp_model_shoulder['easting'].values.max()
SHOULDER_N_MIN = vp_model_shoulder['northing'].values.min()
SHOULDER_N_MAX = vp_model_shoulder['northing'].values.max()

dx = 10

topo_data = topo_data.interp(
    easting=np.arange(SHOULDER_E_MIN, SHOULDER_E_MAX, dx),
    northing=np.arange(SHOULDER_N_MIN, SHOULDER_N_MAX, dx),
)

z = torch.from_numpy(topo_data.values).float()

x_posterior = torch.arange(
    topo_data['easting'].values.min()+dx, topo_data['easting'].values.max()-dx, dx)
y_posterior = torch.arange(
    topo_data['northing'].values.min()+dx, topo_data['northing'].values.max()-dx, dx)
z_posterior = torch.from_numpy(
    topo_data.interp(easting=x_posterior, northing=y_posterior).values)

print(f'Posterior x: {x_posterior.min()} - {x_posterior.max()} ({x_posterior.shape[0]} points)')
print(f'Posterior y: {y_posterior.min()} - {y_posterior.max()} ({y_posterior.shape[0]} points)')
print(f'Posterior z: {z_posterior.min()} - {z_posterior.max()}')

Posterior x: 521.5400390625 - 1901.5400390625 (139 points)
Posterior y: 1141.6800537109375 - 1581.6800537109375 (45 points)
Posterior z: 1935.4761669621366 - 2352.0121661516464


In [22]:
X_posterior, Y_posterior = torch.meshgrid(
    x_posterior, y_posterior, indexing='ij')    

indices = torch.arange(z_posterior.numel())
posterior_grid = torch.stack(
    [X_posterior.ravel(), Y_posterior.ravel(),
     z_posterior.ravel(), indices], dim=-1)

torch.save(posterior_grid, 'data/priors/shoulder_posterior_grid.pt')

In [23]:
prior_dist = concave_hull2D_prior_dist_constructor(
    design_space_shoulder[..., :2],
    topo_data,
    buffer=0.0,
    depth=10.0,
    ratio=0.1,
)

In [24]:
lookup_data = torch.load('data/data_lookup/heterogeneous_shoulder_posterior.pt')

forward_function_heterogenous =  TTLookup(
    posterior_grid, design_space_shoulder, lookup_data,
)

forward_function_shots =  TTLookup(
    events_shoulder, design_space_shoulder,
    torch.load('data/data_lookup/heterogeneous_shoulder_shots.pt'),
)

data_likelihood = DataLikelihood(
    forward_function=forward_function_heterogenous,
    dependence_distance=150,
    vel_sigma=0.05,
    tt_obs_std=0.01,
)

In [25]:

data_likelihood = DataLikelihood(
    forward_function=forward_function_heterogenous,
    dependence_distance=0,
    vel_sigma=0.05,
    tt_obs_std=0.01,
)

In [26]:
design_df = pd.read_csv('generate_designs/data/design_data.csv')
design_df = design_df[design_df['study_area'] == 'shoulder']
design_df = design_df.reset_index(drop=True)

print(f'Number of designs: {len(design_df)}')
print(f'Number of events : {len(events_shoulder)}')

Number of designs: 2226
Number of events : 94


In [27]:
import numpy as np

design_space_shoulder_clean = design_space_shoulder.clone()
# Remove indices_shoulder
faulty_indices = [245, 98, 165, 218, 219]
# Create a mask for indices NOT in faulty_indices
mask = ~torch.isin(design_space_shoulder_clean[:, 3], torch.tensor(faulty_indices, dtype=design_space_shoulder_clean.dtype))
design_space_shoulder_clean = design_space_shoulder_clean[mask]

design_df = design_df.copy()

for idx, row in design_df.iterrows():
    design = np.array(ast.literal_eval(row['design']))
    # Check if any design point has a faulty index
    needs_fixing = False
    for i, design_point in enumerate(design):
        if len(design_point) >= 4 and design_point[3] in faulty_indices:
            needs_fixing = True
        
            # print(f"Faulty index {design_point} found in design at row {idx}, point {i}")
        
            # If it has a faulty index, replace it with the closest point in design_space_shoulder    
            # Find closest point in design_space_shoulder based on first 3 coordinates
            distances = torch.cdist(
                torch.tensor(design_point[:3]).unsqueeze(0).float(),
                design_space_shoulder_clean[:, :3].float()
            )
            closest_idx = distances.argmin().item()
            design[i] = design_space_shoulder_clean[closest_idx].tolist()
    
    # Update the design if any fixes were made
    if needs_fixing:
        design_df.at[idx, 'design'] = str(design.tolist())
        # print(f"Fixed design at row {idx}")

In [28]:
tt_obs_file = 'data/picking_stats/shoulder_tt_obs.pt'

if not os.path.exists(tt_obs_file):
    tt_obs_collection = torch.full((len(events_shoulder), len(design_space_shoulder)), torch.nan)
    print(f'Number of TTs: {tt_obs_collection.numel()}')

    for d_i, design in tqdm(enumerate(design_space_shoulder), total=len(design_space_shoulder)):
        for e_i, event in enumerate(events_shoulder):
            try:
                tt_obs_collection[e_i, d_i] = picking_stats_shoulder[(
                    (picking_stats_shoulder['src_name'] == f'{event[0].item():4.0f}_{event[1].item():4.0f}') &
                    (picking_stats_shoulder['rec_name'] == f'{design[0].item():4.0f}_{design[1].item():4.0f}')
                )]['tt'].values[0]
            except:
                pass
    torch.save(tt_obs_collection, tt_obs_file)
else:
    tt_obs_collection = torch.load(tt_obs_file)


In [29]:
design_space_shoulder.shape

torch.Size([375, 4])

In [30]:
if os.path.exists('benchmark_designs/shoulder/data/summary_statistics.csv'):
    summary_statistics = pd.read_csv('benchmark_designs/shoulder/data/summary_statistics.csv')
else:
    summary_statistics = pd.DataFrame(
        columns=[
            'study_area', 'model_prior', 'velocity_model', 'vel_sigma', 'noise_correlation',
            'drop_mean', 'drop_gradient', 'optimisation', 'EIG_method', 'EIG_N', 'N_rec', 
            'design',
            'mean_x', 'mean_y', 'std_x', 'std_y', 
            'squared_dist_x', 'squared_dist_y', 
            'distance', 'rms_dist_x', 'rms_dist_y',
            'p_posterior', 'posterior_entropy', 'posterior_information',
            'prior_entropy', 'prior_information', 'event', 'event_index'
        ]
    )
    summary_statistics.to_csv('benchmark_designs/shoulder/data/summary_statistics.csv', index=False)

In [31]:
def is_scenario_in_design_data(design_data, scen, N_rec):
    
    design_data = pd.DataFrame(design_data, columns=summary_statistics.columns)
    
    out =  ((design_data['study_area'] == scen['study_area']) &
            (design_data['model_prior'] == scen['model_prior']) &
            (design_data['velocity_model'] == scen['velocity_model']) &
            (design_data['vel_sigma'] == scen['vel_sigma']) &
            (design_data['noise_correlation'] == scen['noise_correlation']) &
            (design_data['drop_mean'] == scen['drop_mean']) &
            (design_data['drop_gradient'] == scen['drop_gradient']) &
            (design_data['optimisation'] == scen['optimisation']) &
            (design_data['EIG_method'] == scen['EIG_method']) &
            (design_data['EIG_N'] == scen['EIG_N']) &
            (design_data['N_rec'] == N_rec)).any()
    return out


In [32]:
with torch.no_grad():
    log_p_prior_original = prior_dist.log_prob(posterior_grid, fast_eval=False)
    log_p_prior_original = log_p_prior_original.reshape(X_posterior.shape)

p_prior = log_p_prior_original.exp()
p_prior = p_prior / torch.nansum(p_prior) # for some reason prior is not normalized, do it here

prior_mask = log_p_prior_original.isfinite()

log_p_prior = p_prior.log()
log_p_prior[~prior_mask] = torch.nan

In [33]:
# for d_i, scen in tqdm(design_df.iterrows(), total=len(design_df)):
    
#     posterior_folder = f'benchmark_designs/shoulder/data/scenario_{scen["study_area"]}_{scen["model_prior"]}_{scen["velocity_model"]}_{scen["vel_sigma"]:.3f}_{scen["noise_correlation"]:.1f}_{scen["drop_mean"]:.1f}_{scen["drop_gradient"]:.1f}_{scen["optimisation"]}_{scen["EIG_method"]}_{scen["EIG_N"]}'
#     os.makedirs(posterior_folder, exist_ok=True)
    
#     design = torch.tensor(ast.literal_eval(scen['design']))
#     receiver_indices = design[..., -1].int().tolist()
    
#     # posterior_stat_file = f'{posterior_folder}/posterior_stats.csv'
    
#     # if os.path.exists(posterior_stat_file):
#     #     posterior_stat_df = pd.read_csv(posterior_stat_file)
#     # else:
#     posterior_stat_df = pd.DataFrame(columns=list(summary_statistics.columns))
#     #     posterior_stat_df.to_csv(posterior_stat_file, index=False)

#     # if is_scenario_in_design_data(posterior_stat_df, scen, len(receiver_indices)):
#     #     # print('Scenario already in design data')
#     #     pass
#     # else:
#     for e_i, event in tqdm(enumerate(events_shoulder), total=len(events_shoulder), desc=f'Scenario {d_i}', leave=False):
        
#         tt_obs = tt_obs_collection[e_i, receiver_indices]
#         design_tmp = design[tt_obs.isfinite()]
#         tt_obs = tt_obs[tt_obs.isfinite()]
        
#         obs_cov = data_likelihood.cov(
#             tt_obs.unsqueeze(0).double(),
#             model_samples    = events_shoulder.unsqueeze(0).double(),
#             design           = design_tmp.double(),
#         )
    
#         if len(tt_obs) < 1:
#             log_p_posterior = log_p_prior.clone()
#         else:
#             log_p_likelihood = data_likelihood(
#                 posterior_grid[prior_mask.flatten()], design_tmp,
#                 # posterior_cov=obs_cov,
#                 ).log_prob(tt_obs)
            
#             log_p_unnormalised_posterior = log_p_likelihood + log_p_prior.flatten()[prior_mask.flatten()]
#             log_p_evidence = torch.logsumexp(log_p_unnormalised_posterior, dim=0)
            
#             log_p_posterior = torch.full_like(log_p_prior, -np.inf).flatten()
#             log_p_posterior[prior_mask.flatten()] = log_p_unnormalised_posterior - log_p_evidence
#             log_p_posterior = log_p_posterior.reshape(X_posterior.shape)
        
#         p_prior = log_p_prior.exp()
#         p_posterior = log_p_posterior.exp()    

#         p_prior = torch.nan_to_num(p_prior, nan=0.0)
#         p_posterior = torch.nan_to_num(p_posterior, nan=0.0)
        
#         # Calculate mean (expected value)
#         mean_x = (X_posterior * p_posterior).sum() / p_posterior.sum()
#         mean_y = (Y_posterior * p_posterior).sum() / p_posterior.sum()
        
#         # Calculate standard deviation
#         std_x = torch.sqrt(((X_posterior - mean_x)**2 * p_posterior).sum() / p_posterior.sum())
#         std_y = torch.sqrt(((Y_posterior - mean_y)**2 * p_posterior).sum() / p_posterior.sum())
        
#         # Calculate distance from mean to true event
#         squared_dist_mean_x = (mean_x - event[0])**2
#         squared_dist_mean_y = (mean_y - event[1])**2
#         distance_mean = torch.sqrt(squared_dist_mean_x + squared_dist_mean_y)
        
#         # Find MAP (maximum a posteriori)
#         max_idx = torch.argmax(p_posterior)
#         map_x = X_posterior.flatten()[max_idx]
#         map_y = Y_posterior.flatten()[max_idx]
        
#         # Calculate distance from MAP to true event
#         squared_dist_map_x = (map_x - event[0])**2
#         squared_dist_map_y = (map_y - event[1])**2
#         distance_map = torch.sqrt(squared_dist_map_x + squared_dist_map_y)
        
#         # Get posterior probability at true event location
#         interp = RegularGridInterpolator(
#             (x_posterior, y_posterior), p_posterior.numpy())
#         post_p = interp(event[:2]).item()
        
#         # Calculate entropy and information
#         entropy_terms = torch.log(p_posterior) * p_posterior
#         post_entropy = -torch.nansum(entropy_terms)
#         post_info = -post_entropy
        
#         entropy_terms_prior = torch.log(p_prior) * p_prior
#         prior_entropy = -torch.nansum(entropy_terms_prior)
#         prior_info = -prior_entropy

#         # prepare row values (same as original insertion)
#         row_values = [
#             scen['study_area'], scen['model_prior'], scen['velocity_model'], scen['vel_sigma'], scen['noise_correlation'],
#             scen['drop_mean'], scen['drop_gradient'], scen['optimisation'], scen['EIG_method'], scen['EIG_N'], scen['N_rec'],
#             design.tolist(),
#             mean_x.item(), mean_y.item(), std_x.item(), std_y.item(),
#             distance_mean.item(), distance_map.item(),
#             post_p, post_entropy.item(), post_info.item(), prior_entropy.item(), prior_info.item(),
#             event, e_i
#         ]


#     # if d_i > 100:
#     #     break


#     # posterior_stat_df.to_csv(posterior_stat_file, index=False)
    

In [34]:
from ipywidgets import interact, IntSlider



def plot_posterior(design_index, event_index, fig_size=(10, 8)):
    """
    Plots the posterior distribution for a given design and event.
    
    Parameters:
    -----------
    design_index : int
        Index of the design in design_df
    event_index : int
        Index of the event in events_shoulder
    fig_size : tuple
        Figure size (width, height)
    """
    # Get design information
    scen = design_df.iloc[design_index]
    design = torch.tensor(ast.literal_eval(scen['design']))
    receiver_indices = design[..., -1].int().tolist()
    
    # Get event information
    event = events_shoulder[event_index]
    
    # Get observed travel times
    tt_obs = tt_obs_collection[event_index, receiver_indices]
    design_tmp = design[tt_obs.isfinite()]
    tt_obs = tt_obs[tt_obs.isfinite()]
    
    # Compute covariance matrix
    obs_cov = data_likelihood.cov(
        tt_obs.unsqueeze(0).double(),
        model_samples=event.unsqueeze(0).double(),
        design=design_tmp.double(),
    )
    
    print(tt_obs)

    print(obs_cov)

    # Calculate posterior
    if len(tt_obs) < 1:
        log_p_posterior = log_p_prior.clone()
    else:
        log_p_likelihood = data_likelihood(
            posterior_grid[prior_mask.flatten()], design_tmp,
            # posterior_cov=obs_cov,
        ).log_prob(tt_obs)
        
        log_p_unnormalised_posterior = log_p_likelihood + log_p_prior.flatten()[prior_mask.flatten()]
        log_p_evidence = torch.logsumexp(log_p_unnormalised_posterior, dim=0)
        
        log_p_posterior = torch.full_like(log_p_prior, -np.inf).flatten()
        log_p_posterior[prior_mask.flatten()] = log_p_unnormalised_posterior - log_p_evidence
        log_p_posterior = log_p_posterior.reshape(X_posterior.shape)
    
    p_posterior = log_p_posterior.exp()
    p_posterior = torch.nan_to_num(p_posterior, nan=0.0)
    
    # Create the figure
    fig, ax = plt.subplots(1, 1, figsize=fig_size, dpi=120)
    
    # Plot posterior probability as contour
    contour = ax.contourf(X_posterior.numpy(), Y_posterior.numpy(), p_posterior.numpy(), 
                          cmap='Reds', alpha=0.7, levels=50)
    
    # Plot the receiver locations with travel time data
    for i, (rx, valid) in enumerate(zip(design, tt_obs_collection[event_index, receiver_indices].isfinite())):
        if valid:
            travel_time = tt_obs_collection[event_index, receiver_indices[i]].item()
            ax.scatter(rx[0], rx[1], c='red', marker='^', s=100, 
                   edgecolor='black', zorder=5)
            ax.annotate(f'{travel_time:.3f}s', (rx[0], rx[1]), 
                     xytext=(5, 5), textcoords='offset points', fontsize=8)
        else:
            ax.scatter(rx[0], rx[1], c='lightgray', marker='^', s=100, 
                   edgecolor='black', zorder=4)
            ax.annotate('no data', (rx[0], rx[1]), 
                     xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Plot the true event location
    ax.scatter(event[0], event[1], c='yellow', marker='*', s=200, 
               label='True Event', edgecolor='black', zorder=10)
    
    # Calculate statistics for annotation
    mean_x = (X_posterior * p_posterior).sum() / p_posterior.sum()
    mean_y = (Y_posterior * p_posterior).sum() / p_posterior.sum()
    
    # Plot the mean of the posterior
    ax.scatter(mean_x, mean_y, c='cyan', marker='x', s=100,
               label='Posterior Mean', zorder=8)
    
    # Calculate distance
    squared_dist_x = ((X_posterior - event[0])**2 * p_posterior).sum() / p_posterior.sum()
    squared_dist_y = ((Y_posterior - event[1])**2 * p_posterior).sum() / p_posterior.sum()
    distance = torch.sqrt(squared_dist_x + squared_dist_y)
        
    # Set labels and title
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(f'Posterior Distribution - Design {design_index}, Event {event_index}\n'
                f'N receivers: {len(design)}, Valid TTs: {len(tt_obs)}, Loc. Error: {distance.item():.1f}m')
    
    # Add grid and legend
    ax.set_aspect('equal')
    ax.legend(loc='upper right')
    
    # Set axis limits to match the posterior grid
    ax.set_xlim(X_posterior.min(), X_posterior.max())
    ax.set_ylim(Y_posterior.min(), Y_posterior.max())
    
    plt.tight_layout()
    
    plt.show()


def update_plot(design_index, event_index):
    plot_posterior(design_index, event_index)

interact(
    update_plot,
    design_index=IntSlider(min=0, max=len(design_df)-1, step=1, value=751, description='Design:'),
    event_index=IntSlider(min=0, max=len(events_shoulder)-1, step=1, value=42, description='Event:')
)

interactive(children=(IntSlider(value=751, description='Design:', max=2225), IntSlider(value=42, description='…

<function __main__.update_plot(design_index, event_index)>

In [21]:
design_df[
    (design_df['study_area'] == 'shoulder') &
    (design_df['model_prior'] == 'uniform') &
    (design_df['velocity_model'] == 'gradient') &
    (design_df['vel_sigma'] == 0.005) &
    (design_df['noise_correlation'] == 100.0) &
    
    # (design_df['drop_mean'] == 0.35) &
    # (design_df['drop_gradient'] == -30.0) &
    
    (design_df['drop_mean'] == 0.0) &
    (design_df['drop_gradient'] == 0.0) &
    
    (design_df['optimisation'] == 'genetic') &
    (design_df['EIG_method'] == 'NMC') &
    (design_df['EIG_N'] == 1000) &
    (design_df['N_rec'] == 9)
].head(10)
    
    

,study_area,model_prior,velocity_model,vel_sigma,noise_correlation,drop_mean,drop_gradient,optimisation,EIG_method,N_rec,design,EIG,EIG_ref,EIG_N,runtime,EIG_candidates
474,shoulder,uniform,gradient,0.005,100,0.0,0,genetic,NMC,9,"[[988.06298828125, 1550.9959716796875, 2296.79...",5.195922,5.228731,1000,4593.092365,"[5.055568695068359, 5.064677715301514, 5.06467..."


In [ ]:
# def plot_event_data(event_index, design_index=None):
#     fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
#     # Get travel times for the current event
#     tt_data = tt_obs_collection[event_index, :]
    
#     # Create a colormap that handles NaN values
#     scatter = ax.scatter(
#         design_space_shoulder[:, 0], design_space_shoulder[:, 1],
#         c=tt_data, s=20, cmap='viridis', alpha=0.8,
#         vmin=0.0, vmax=0.4
#     )
    
#     # Plot the event location with a star
#     ax.scatter(
#         events_shoulder[event_index][0], events_shoulder[event_index][1],
#         facecolor='red', s=100, marker='*', label='Event', edgecolor='black'
#     )
    
#     if design_index is not None:
#         # Plot the design location with a triangle
#         scen = design_df.iloc[design_index]
#         design = torch.tensor(ast.literal_eval(scen['design']))
        
#         ax.scatter(
#             design[..., 0], design[..., 1],
#             marker='*', s=100, facecolor='black',
#             label='Design', edgecolor='black', zorder=-1)
    
#     # Calculate percentage of valid travel times
#     valid_tt = (~torch.isnan(tt_data)).sum().item()
#     total_tt = tt_data.numel()
#     valid_percentage = (valid_tt / total_tt) * 100
    
#     ax.set_aspect('equal')
#     ax.set_xlabel('Easting (m)')
#     ax.set_ylabel('Northing (m)')
#     ax.set_title(f'Event {event_index} - Travel Time Observations\n'
#                 f'Valid observations: {valid_tt}/{total_tt} ({valid_percentage:.1f}%)')
    
#     ax.legend()
    
#     cbar = plt.colorbar(scatter, ax=ax, label='Travel Time (s)', shrink=0.6)
    
#     ax.set_xlim(SHOULDER_E_MIN, SHOULDER_E_MAX)
#     ax.set_ylim(SHOULDER_N_MIN, SHOULDER_N_MAX)
    
#     plt.tight_layout()
#     plt.show()
    
#     return fig

# design_index = 95
# event_index = 42

# plot_event_data(event_index, design_index=design_index);

In [ ]:
# def plot_travel_time_field(event_index, design_index=None, figsize=(8, 6)):
#     """
#     Plots travel time field from a source event to the entire grid,
#     highlighting design points.
    
#     Parameters:
#     -----------
#     event_index : int
#         Index of the event in events_shoulder
#     design_index : int, optional
#         Index of the design in design_df
#     fig_size : tuple, optional
#         Figure size (width, height)
#     """
#     # Get event information
#     event = events_shoulder[event_index]
#     print(event)
    
#     # Create the figure
#     fig, ax = plt.subplots(1, 1, figsize=figsize)
        
#     travel_times = forward_function_shots(
#         event.unsqueeze(0).double(),
#         design_space_shoulder,
#     )
#     sc = ax.scatter(
#         design_space_shoulder[:, 0], design_space_shoulder[:, 1],
#         c=travel_times, s=20, cmap='viridis', alpha=0.8, vmin=0.0, vmax=0.4
#     )
    
    
#     # tt_data = lookup_data[..., event_index]
#     # sc = ax.scatter(
#     #     posterior_grid[:, 0], posterior_grid[:, 1],
#     #     c=tt_data, s=20, cmap='viridis', alpha=0.8, vmin=0.15, vmax=0.28
#     # )

#     # Plot the event location with a star
#     ax.scatter(event[0], event[1], c='yellow', marker='*', s=200, 
#               label='Event Source', edgecolor='black', zorder=10,
#               )
    
#     # If a design is specified, plot the receivers
#     if design_index is not None:
#         scen = design_df.iloc[design_index]
#         design = torch.tensor(ast.literal_eval(scen['design']))
        
#         # Get travel times to each receiver in the design
#         tt_obs = tt_obs_collection[event_index, design[..., -1].int().tolist()]
        
#         # Plot each receiver in the design, with travel time if available
#         for i, (rx, tt_valid) in enumerate(zip(design, tt_obs.isfinite())):
#             if tt_valid:
#                 travel_time = tt_obs[i].item()
#                 ax.scatter(rx[0], rx[1], c='red', marker='^', s=100, 
#                           edgecolor='black', zorder=5, label='Receiver (valid data)' if i == 0 else "")
#                 ax.annotate(f'{travel_time:.3f}s', (rx[0], rx[1]), 
#                           xytext=(5, 5), textcoords='offset points', fontsize=8)
#             else:
#                 ax.scatter(rx[0], rx[1], c='lightgray', marker='^', s=100, 
#                           edgecolor='black', zorder=4, label='Receiver (no data)' if i == 0 else "")
    
#     # Set labels and title
#     ax.set_xlabel('Easting (m)')
#     ax.set_ylabel('Northing (m)')
#     ax.set_title(f'Travel Time Field - Event {event_index}')
    
#     # Create a colorbar
#     cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
#     cbar.set_label('Travel Time (s)')
    
#     # Set boundaries
#     ax.set_xlim(SHOULDER_E_MIN, SHOULDER_E_MAX)
#     ax.set_ylim(SHOULDER_N_MIN, SHOULDER_N_MAX)
    
#     ax.set_aspect('equal')
    
#     return fig


# design_index = 95
# event_index = 42

# plot_travel_time_field(event_index, design_index=design_index);

In [ ]:
# def plot_travel_time_comparison(event_index, design_index=None, figsize=(15, 5)):
#     """
#     Creates a three-panel plot showing:
#     1. Observed travel times
#     2. Forward model predictions
#     3. Difference between observed and predicted times
    
#     Parameters:
#     -----------
#     event_index : int
#         Index of the event in events_shoulder
#     design_index : int, optional
#         Index of the design in design_df
#     figsize : tuple, optional
#         Figure size (width, height)
#     """
#     # Get event information
#     event = events_shoulder[event_index]
    
#     # Create a figure with three subplots
#     fig, axes = plt.subplots(1, 3, figsize=figsize)
    
#     # Define colormap range for consistency
#     vmin, vmax = 0.0, 0.4
#     diff_max = 0.05  # Max difference for the colorbar
    
#     # --- Panel 1: Observed travel times ---
#     ax = axes[0]
#     # Get travel times for the current event
#     tt_data = tt_obs_collection[event_index, :]
    
#     # Create a colormap that handles NaN values
#     sc1 = ax.scatter(
#         design_space_shoulder[:, 0], design_space_shoulder[:, 1],
#         c=tt_data, s=20, cmap='viridis', alpha=0.8,
#         vmin=vmin, vmax=vmax,  zorder=11,
#     )
    
#     # Plot the event location
#     ax.scatter(
#         event[0], event[1], 
#         facecolor='red', s=100, marker='*', 
#         label='Event', edgecolor='black', zorder=10
#     )
    
#     # If a design is specified, plot the design points
#     if design_index is not None:
#         scen = design_df.iloc[design_index]
#         design = torch.tensor(ast.literal_eval(scen['design']))
        
#         ax.scatter(
#             design[..., 0], design[..., 1],
#             marker='v', s=80, facecolor='white',
#             label='Design', edgecolor='black', zorder=5)
    
#     ax.set_aspect('equal')
#     ax.set_xlabel('Easting (m)')
#     ax.set_ylabel('Northing (m)')
#     ax.set_title('Observed Travel Times')
    
#     ax.set_xlim(SHOULDER_E_MIN, SHOULDER_E_MAX)
#     ax.set_ylim(SHOULDER_N_MIN, SHOULDER_N_MAX)
    
#     cbar1 = plt.colorbar(sc1, ax=ax, label='Travel Time (s)', shrink=0.7, orientation='horizontal')
    
#     # --- Panel 2: Forward model predictions ---
#     ax = axes[1]
    
#     # Generate forward model predictions
#     with torch.no_grad():
#         travel_times = forward_function_shots(
#             event.unsqueeze(0).double(),
#             design_space_shoulder,
#         )
    
#     sc2 = ax.scatter(
#         design_space_shoulder[:, 0], design_space_shoulder[:, 1],
#         c=travel_times, s=20, cmap='viridis', alpha=0.8,
#         vmin=vmin, vmax=vmax,  zorder=11,
#     )
    
#     # Plot the event location
#     ax.scatter(
#         event[0], event[1], 
#         facecolor='red', s=100, marker='*', 
#         label='Event', edgecolor='black', zorder=10
#     )
    
#     # If a design is specified, plot the design points
#     if design_index is not None:
#         ax.scatter(
#             design[..., 0], design[..., 1],
#             marker='v', s=80, facecolor='white',
#             label='Design', edgecolor='black', zorder=5)
    
#     ax.set_aspect('equal')
#     ax.set_xlabel('Easting (m)')
#     ax.set_title('Predicted Travel Times')
    
#     ax.set_xlim(SHOULDER_E_MIN, SHOULDER_E_MAX)
#     ax.set_ylim(SHOULDER_N_MIN, SHOULDER_N_MAX)
    
#     cbar2 = plt.colorbar(sc2, ax=ax, label='Travel Time (s)', shrink=0.7, orientation='horizontal')
    
#     # --- Panel 3: Difference between observed and predicted ---
#     ax = axes[2]
    
#     # Calculate differences (observed - predicted)
#     tt_diff = tt_data - travel_times
    
#     # Use a diverging colormap for the difference
#     sc3 = ax.scatter(
#         design_space_shoulder[:, 0], design_space_shoulder[:, 1],
#         c=tt_diff, s=20, cmap='RdBu_r', alpha=0.8,
#         vmin=-0.02, vmax=0.02, zorder=11,
#     )
    
#     # Plot the event location
#     ax.scatter(
#         event[0], event[1], 
#         facecolor='red', s=100, marker='*', 
#         label='Event', edgecolor='black', zorder=10
#     )
    
#     # If a design is specified, plot the design points
#     if design_index is not None:
#         ax.scatter(
#             design[..., 0], design[..., 1],
#             marker='v', s=80, facecolor='white',
#             label='Design', edgecolor='black', zorder=5)
    
#     ax.set_aspect('equal')
#     ax.set_xlabel('Easting (m)')
#     ax.set_title('Difference (Observed - Predicted)')
    
#     ax.set_xlim(SHOULDER_E_MIN, SHOULDER_E_MAX)
#     ax.set_ylim(SHOULDER_N_MIN, SHOULDER_N_MAX)
    
#     cbar3 = plt.colorbar(sc3, ax=ax, label='Time Difference (s)', shrink=0.7, orientation='horizontal')
    
#     # Calculate some statistics for the non-NaN differences
#     valid_diff = tt_diff[~torch.isnan(tt_diff)]
#     if len(valid_diff) > 0:
#         mean_diff = valid_diff.mean().item()
#         std_diff = valid_diff.std().item()
#         rms_diff = torch.sqrt((valid_diff**2).mean()).item()
        
#         # Add text with statistics to the subplot
#         ax.text(0.05, 0.05, 
#                 f'Mean: {mean_diff:.3f}s\nStd: {std_diff:.3f}s\nRMS: {rms_diff:.3f}s',
#                 transform=ax.transAxes, 
#                 bbox=dict(facecolor='white', alpha=0.7))
    
#     # Add common legend
#     handles, labels = axes[0].get_legend_handles_labels()
#     fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2)
    
#     plt.tight_layout()
    
#     return fig

# design_index = 95
# event_index = 42

# plot_travel_time_comparison(event_index, design_index=design_index);

In [ ]:
# Initialize a list to store all dataframes
all_dfs = []

# Look for posterior_stat files in the benchmark_designs/shoulder/data directories
for root, dirs, files in os.walk('benchmark_designs/shoulder/data'):
    for dir_name in dirs:
        if dir_name.startswith('scenario_'):
            posterior_stat_file = os.path.join(root, dir_name, 'posterior_stats.csv')
            if os.path.exists(posterior_stat_file):
                try:
                    # Read the dataframe
                    df = pd.read_csv(posterior_stat_file)
                                    
                    all_dfs.append(df)
                    # print(f"Processed {posterior_stat_file}")
                except Exception as e:
                    print(f"Error processing {posterior_stat_file}: {e}")

# Combine all dataframes
if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # Update the summary_statistics dataframe
    summary_statistics = combined_df
    
    # Save the updated summary statistics
    summary_statistics.to_csv('benchmark_designs/shoulder/data/summary_statistics.csv', index=False)
    
    print(f"Combined {len(all_dfs)} posterior statistic files into summary_statistics.csv")
    print(f"Total number of entries: {len(summary_statistics)}")
else:
    print("No posterior statistic files found to process.")


In [ ]:
summary_statistics

In [ ]:
scen_columns = [
    'study_area', 'model_prior', 'velocity_model', 'vel_sigma', 'noise_correlation',
    'drop_mean', 'drop_gradient', 'optimisation', 'EIG_method', 'EIG_N', 'N_rec', 'design'
]
stat_columns = [
    'mean_x', 'mean_y',
    'squared_dist_x', 'squared_dist_y',
    'distance', 'rms_dist_x', 'rms_dist_y',
    'p_posterior', 'posterior_entropy', 'posterior_information',
    'prior_entropy', 'prior_information'
]       

In [ ]:
# Group by all parameters except event and event_index
grouped = summary_statistics.groupby(scen_columns)[stat_columns].mean().reset_index()

grouped = grouped[(grouped['drop_mean'] == 0) | (grouped['drop_mean'] == 0.35)]
grouped = grouped[(grouped['drop_gradient'] == 0.0) | (grouped['drop_gradient'] == -30.0)]
grouped = grouped[(grouped['optimisation'] == 'genetic')]
grouped = grouped[(grouped['vel_sigma'] == 0.05)]
grouped = grouped[(grouped['N_rec'] <= 15)]
grouped = grouped[(grouped['noise_correlation'] == 100)]
grouped = grouped[(grouped['velocity_model'] == 'heterogeneous')]
grouped = grouped[(grouped['EIG_N'] == 1000)]
grouped = grouped[(grouped['EIG_method'] == 'NMC')]

In [ ]:
grouped.columns

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Create a figure with a smaller size
fig, ax = plt.subplots(figsize=(8, 6))

# Define markers and line styles
markers = {0.0: 'o', 0.35: '^'}  # Different markers for drop_mean values
linestyles = {'NMC': '-', 'DN': '--'}  # Different line styles for methods

# Define EIG_N values and create a color gradient using rainbow colormap
eig_n_values = sorted(grouped['EIG_N'].unique())
eig_n_colors = {val: plt.cm.rainbow(i/len(eig_n_values)) for i, val in enumerate(eig_n_values)}

# Plot data for each combination of parameters
for eig_method in sorted(grouped['EIG_method'].unique()):
    for drop_mean in sorted(grouped['drop_mean'].unique()):
        for eig_n in sorted(grouped['EIG_N'].unique()):
            subset = grouped[(grouped['EIG_method'] == eig_method) & 
                             (grouped['drop_mean'] == drop_mean) &
                             (grouped['EIG_N'] == eig_n)]
            
            if not subset.empty:
                # Sort by N_rec to ensure proper line connections
                subset = subset.sort_values('N_rec')
                
                # Plot scatter points with smaller marker size
                ax.scatter(subset['N_rec'], subset['distance'], 
                           color=eig_n_colors[eig_n],
                           marker=markers[drop_mean],
                           s=50, alpha=0.8,
                           label=f"{eig_method}, Drop={drop_mean}, EIG_N={eig_n}")
                
                # # Connect points with thinner lines
                ax.plot(subset['N_rec'], subset['distance'], 
                       color=eig_n_colors[eig_n], 
                       linestyle=linestyles[eig_method],
                       alpha=0.7,
                       linewidth=1.5)

# Add labels and title with smaller fontsize
ax.set_title('Mean Localization Error vs Number of Receivers', fontsize=14)
ax.set_xlabel('Number of Receivers', fontsize=12)
ax.set_ylabel('Mean Distance (m)', fontsize=12)

# Create custom legend elements
legend_elements = []

# Method legend (linestyles)
for method, ls in linestyles.items():
    legend_elements.append(Line2D([0], [0], color='black', 
                                 linestyle=ls, lw=1.5,
                                 label=f'Method: {method}'))

# Drop mean legend (markers)
for drop, marker in markers.items():
    legend_elements.append(Line2D([0], [0], marker=marker, color='black', 
                                 linestyle='None', markersize=6, 
                                 label=f'Drop Mean: {drop}'))

# EIG_N legend (colors)
for eig_n, color in eig_n_colors.items():
    legend_elements.append(Line2D([0], [0], color=color, lw=1.5, 
                                 label=f'EIG_N: {eig_n}'))

# Add grid and legend with smaller fontsize
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(handles=legend_elements, loc='upper right', fontsize=8)

# Set x-axis to show whole numbers
ax.set_xticks(sorted(grouped['N_rec'].unique()))
ax.tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.show()